In [ ]:
from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaPlanning

from tapas_gmm.dataset.scene import SceneDataset, SceneDatasetConfig
from pathlib import Path
import torch

In [ ]:
data_root = Path("../outputs/bimanual_dataset")

In [ ]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode=BimanualEndEffectorPoseViaPlanning,
        robot_setup="dual_panda",
        task="BimanualDualPushButtons",
        cameras=tuple(),
        camera_pose={},
        image_size=(128, 128),
        static=False,
        headless=True,
        scale_action=False,
        delay_gripper=False,
        gripper_plot=False,   
        absolute_action_mode=False,
        action_frame="end effector",
    )
)

In [ ]:
all_demos = []

while len(all_demos) < 5:
    try:
        demo = tapas_env.task_env.get_demos(1, live_demos=True)
        all_demos.append(demo[0])
        print("success")

    except RuntimeError as e:
        print(e)

demos = all_demos

In [ ]:
dataset_config = SceneDatasetConfig(
    data_root=data_root,
    camera_names=tuple(),
    image_size=(128, 128),
)

scene_dataset = SceneDataset(
    allow_creation=True,
    data_root=data_root,
    config=dataset_config,
)

In [ ]:
for demo_index, demo in enumerate(demos):
    for i in range(len(demo) - 1):
        raw_obs = demo[i]
        next_raw_obs = demo[i + 1]
        action = tapas_env._get_action(raw_obs, next_raw_obs)
        scene_observation = tapas_env.process_observation(raw_obs)
        scene_observation.action = torch.Tensor(action)
        scene_observation.feedback = torch.Tensor([1])
        scene_dataset.add_observation(scene_observation)

    scene_dataset.save_current_traj(traj_suffix=f"_{demo_index}")
        

In [ ]:
tapas_env.env.shutdown()